In [1]:
import pandas as pd
import numpy as np

!pip install catboost
!pip install imbalanced-learn -U
!pip install specificity

# avoid warning
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category=ConvergenceWarning)


from sklearn.utils import shuffle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier , BaggingClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [2]:
# Load dataset
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [3]:

# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})





In [4]:
# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

# 7. Prepare your data
X_raw = df.drop('y', axis=1)
y_raw = df['y']


In [5]:
# 8. Apply SMOTE
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_shuffled, y_shuffled = sm.fit_resample(X_raw, y_raw)

In [6]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform



# Split
X_train, X_test, y_train, y_test = train_test_split(X_shuffled, y_shuffled, test_size=0.2, random_state=42)

# Model
model = LogisticRegression(max_iter=1000)

# Search space
param_dist = {
    "C": uniform(0.01, 10),
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [500, 1000, 2000],

}

# RandomizedSearchCV
random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=10,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'C': np.float64(6.128528947223795), 'max_iter': 1000, 'solver': 'liblinear'}
Best CV accuracy: 0.8556
Test accuracy: 0.8547


In [7]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Logistic Regression)
from sklearn.linear_model import LogisticRegression

clf_name = "Logistic Regression"
clf = LogisticRegression(random_state=42, max_iter=1000)



# Store results
results = []

# KFold Cross-Validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results_df = pd.DataFrame(results)

# Print
print(f"Results for {clf_name}:")
print(results_df)

# Save to CSV if needed
results_df.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Logistic Regression:
   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Logistic Regression  0.852394   0.880588  0.816471     0.888553   
1     2  Logistic Regression  0.839808   0.862212  0.805302     0.873679   
2     3  Logistic Regression  0.848564   0.864514  0.824807     0.872074   
3     4  Logistic Regression  0.848290   0.863006  0.829886     0.866887   
4     5  Logistic Regression  0.854309   0.886035  0.815489     0.893664   
5     6  Logistic Regression  0.853078   0.875508  0.823867     0.882401   
6     7  Logistic Regression  0.847312   0.881720  0.810580     0.885867   
7     8  Logistic Regression  0.847038   0.870280  0.818823     0.875759   
8     9  Logistic Regression  0.853742   0.865616  0.834300     0.872831   
9    10  Logistic Regression  0.849090   0.870695  0.810233     0.885928   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.847319  0.851750  0.111447  0.852512  0.706743  0.704856   


In [8]:
# Model
from sklearn.tree import DecisionTreeClassifier
model1 = DecisionTreeClassifier()

# Search space
param_dist1 = {

    "max_depth":[5,10,20,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

# DecisionTreeClassifier

random_search = RandomizedSearchCV(model1, param_distributions=param_dist1, n_iter=50,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 36 is smaller than n_iter=50. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Best CV accuracy: 0.6119
Test accuracy: 0.9005


In [9]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Decision Tree)
from sklearn.tree import DecisionTreeClassifier
clf_name = "Decision Tree"
clf = DecisionTreeClassifier()



# Store results
results1 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results1.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results1_df1 = pd.DataFrame(results1)

# Print
print(f"Results for {clf_name}:")
print(results1_df1)

# Save to CSV if needed
results1_df1.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Decision Tree:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Decision Tree  0.896443   0.887587  0.908645     0.884161  0.897992   
1     2  Decision Tree  0.899316   0.885798  0.914664     0.884250  0.900000   
2     3  Decision Tree  0.901778   0.888445  0.917767     0.885955  0.902868   
3     4  Decision Tree  0.901368   0.887838  0.919978     0.882563  0.903623   
4     5  Decision Tree  0.906703   0.902308  0.913587     0.899725  0.907913   
5     6  Decision Tree  0.901094   0.889272  0.916712     0.885417  0.902783   
6     7  Decision Tree  0.905049   0.900447  0.915843     0.893718  0.908079   
7     8  Decision Tree  0.904228   0.892716  0.920803     0.887355  0.906542   
8     9  Decision Tree  0.905596   0.893634  0.918807     0.892625  0.906046   
9    10  Decision Tree  0.899850   0.886034  0.911442     0.888859  0.898559   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.896319  0.115

In [10]:
# Model
from sklearn.ensemble import RandomForestClassifier
model2 = RandomForestClassifier()

# Search space
param_dist2 = {
    "n_estimators":[100,200,300],
    "max_depth":[10,20,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}

In [11]:
# RandomForestClassifier

random_search = RandomizedSearchCV(model2, param_distributions=param_dist2, n_iter=15,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Best CV accuracy: 0.6853
Test accuracy: 0.9398


In [12]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Random Forest)
from sklearn.ensemble import RandomForestClassifier
clf_name = "Random Forest"
clf = RandomForestClassifier()



# Store results
results2 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results2.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results2_df2 = pd.DataFrame(results2)

# Print
print(f"Results for {clf_name}:")
print(results2_df2)

# Save to CSV if needed
results2_df2.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Random Forest:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.938577   0.942276  0.934824     0.942355  0.938535   
1     2  Random Forest  0.930096   0.928139  0.930958     0.929249  0.929546   
2     3  Random Forest  0.944596   0.943210  0.945545     0.943658  0.944376   
3     4  Random Forest  0.937346   0.931330  0.945019     0.929593  0.938125   
4     5  Random Forest  0.938030   0.944613  0.931522     0.944628  0.938022   
5     6  Random Forest  0.938167   0.934724  0.942381     0.933936  0.938537   
6     7  Random Forest  0.940621   0.946800  0.936682     0.944756  0.941714   
7     8  Random Forest  0.933780   0.931322  0.937890     0.929597  0.934595   
8     9  Random Forest  0.939116   0.934830  0.942833     0.935466  0.938815   
9    10  Random Forest  0.938979   0.936816  0.937869     0.940032  0.937342   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.938582  0.057

In [13]:
# Model
from sklearn.neighbors import KNeighborsClassifier
model3 = KNeighborsClassifier()

# Search space
param_dist3 = {
         "n_neighbors": [3, 5, 7],
            "weights": ["uniform", "distance"]
}

In [14]:
# KNeighbors Classifier

random_search = RandomizedSearchCV(model3, param_distributions=param_dist3, n_iter=25,
                                   cv=15, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=25. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'weights': 'distance', 'n_neighbors': 3}
Best CV accuracy: 0.8427
Test accuracy: 0.9227


In [15]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (KNN)
from sklearn.neighbors import KNeighborsClassifier
clf_name = "KNN"
clf = KNeighborsClassifier()

# Store results
results3 = []




# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results3.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results3_df3 = pd.DataFrame(results3)

# Print
print(f"Results for {clf_name}:")
print(results3_df3)

# Save to CSV if needed
results3_df3.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for KNN:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        KNN  0.910397   0.875000  0.958277     0.862201  0.914747   
1     2        KNN  0.901368   0.861776  0.953880     0.849824  0.905492   
2     3        KNN  0.913680   0.878750  0.958746     0.869080  0.917006   
3     4        KNN  0.908345   0.872335  0.957812     0.858361  0.913077   
4     5        KNN  0.915321   0.881006  0.961685     0.868320  0.919579   
5     6        KNN  0.910944   0.878361  0.954397     0.867325  0.914802   
6     7        KNN  0.913805   0.884606  0.956452     0.869041  0.919127   
7     8        KNN  0.910521   0.878274  0.954977     0.865268  0.915021   
8     9        KNN  0.917499   0.880484  0.964374     0.871475  0.920522   
9    10        KNN  0.910795   0.873106  0.955581     0.868337  0.912483   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.908971  0.137799  0.910239  0.824505  0.820735           0.910239  

In [16]:

# Model
from sklearn.svm import LinearSVC
model4 = LinearSVC()



from sklearn.svm import LinearSVC

model4 = LinearSVC()

param_dist4 = {
    "C": [0.01, 0.1, 1, 10],
    "loss": ["hinge", "squared_hinge"],
    "max_iter": [1000, 2000]
}

In [17]:

# LinearSVC

random_search = RandomizedSearchCV(model4, param_distributions=param_dist4, n_iter=10,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))


Best parameters: {'max_iter': 1000, 'loss': 'squared_hinge', 'C': 0.01}
Best CV accuracy: 0.7626
Test accuracy: 0.8499


In [18]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (LinearSVC)
from sklearn.svm import LinearSVC
clf_name = "SVM"
clf = LinearSVC()


# Store results
results4 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results4.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results4_df4 = pd.DataFrame(results4)

# Print
print(f"Results for {clf_name}:")
print(results4_df4)

# Save to CSV if needed
results4_df4.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for SVM:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        SVM  0.858413   0.889119  0.820016     0.897063  0.853171   
1     2        SVM  0.850342   0.877728  0.810826     0.889130  0.842951   
2     3        SVM  0.855677   0.876568  0.826183     0.884867  0.850630   
3     4        SVM  0.851436   0.875290  0.821448     0.881738  0.847515   
4     5        SVM  0.856908   0.891033  0.815489     0.898898  0.851589   
5     6        SVM  0.858413   0.883280  0.826597     0.890351  0.853999   
6     7        SVM  0.855384   0.898280  0.809244     0.903814  0.851441   
7     8        SVM  0.853058   0.879245  0.821535     0.885146  0.849411   
8     9        SVM  0.860856   0.881818  0.830434     0.890727  0.855355   
9    10        SVM  0.849501   0.874886  0.806016     0.890725  0.839040   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.857675  0.102937  0.858540  0.719084  0.716896           0.858540  

In [19]:
# Model

from xgboost import XGBClassifier

model5 =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')

param_dist5 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1],
            "max_depth": [3, 6]
}

In [20]:

# XGBoost

random_search = RandomizedSearchCV(model5, param_distributions=param_dist5, n_iter=100,
                                   cv=20, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=100. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best parameters: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1}
Best CV accuracy: 0.8121
Test accuracy: 0.8774


In [21]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from xgboost import XGBClassifier
clf_name = "XGBoost"
clf =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')


# Store results
results5 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results5.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results5_df5 = pd.DataFrame(results5)

# Print
print(f"Results for {clf_name}:")
print(results5_df5)

# Save to CSV if needed
results5_df5.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:45] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

Results for XGBoost:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1    XGBoost  0.911354   0.941503  0.877829     0.945100  0.908552   
1     2    XGBoost  0.904241   0.935320  0.866611     0.941176  0.899656   
2     3    XGBoost  0.915732   0.942296  0.884763     0.946380  0.912624   
3     4    XGBoost  0.916416   0.935456  0.895482     0.937569  0.915033   
4     5    XGBoost  0.915458   0.948448  0.879891     0.951515  0.912884   
5     6    XGBoost  0.914090   0.938186  0.886947     0.941338  0.911847   
6     7    XGBoost  0.910111   0.950117  0.870158     0.952047  0.908381   
7     8    XGBoost  0.911889   0.939359  0.882289     0.942021  0.909930   
8     9    XGBoost  0.916952   0.939103  0.890086     0.943330  0.913937   
9    10    XGBoost  0.908059   0.935932  0.870678     0.943497  0.902126   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.910844  0.054900  0.911465  0.824646  0.822746           0.9114

In [22]:
# Model LightGBM



model6 =  LGBMClassifier()

param_dist6 = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.1]
}

In [23]:

# LightGBM

random_search = RandomizedSearchCV(model6, param_distributions=param_dist6, n_iter=10,
                                   cv=3, scoring="accuracy", random_state=42,verbose=1, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 3 folds for each of 4 candidates, totalling 12 fits
[LightGBM] [Info] Number of positive: 32991, number of negative: 32796
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043577 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1047
[LightGBM] [Info] Number of data points in the train set: 65787, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501482 -> initscore=0.005928
[LightGBM] [Info] Start training from score 0.005928
Best parameters: {'n_estimators': 200, 'learning_rate': 0.1}
Best CV accuracy: 0.4664
Test accuracy: 0.9111


In [24]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from lightgbm import LGBMClassifier
clf_name = "LightGBM"
clf =  LGBMClassifier()


# Store results
results6 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results6.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results6_df6 = pd.DataFrame(results6)

# Print
print(f"Results for {clf_name}:")
print(results6_df6)

# Save to CSV if needed
results6_df6.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

[LightGBM] [Info] Number of positive: 32881, number of negative: 32905
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 65786, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499818 -> initscore=-0.000730
[LightGBM] [Info] Start training from score -0.000730
[LightGBM] [Info] Number of positive: 32927, number of negative: 32859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031782 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1050
[LightGBM] [Info] Number of data points in the train set: 65786, number of used features: 45
[LightGBM] [Info] 

In [25]:
# Model CatBoost

from catboost import CatBoostClassifier

model7 =   CatBoostClassifier(verbose=0)

param_dist7 = {
     "iterations": [100, 200],
     "depth": [4, 6],
     "learning_rate": [0.01, 0.1]
}

In [26]:
# LightGBM

random_search = RandomizedSearchCV(model7, param_distributions=param_dist7, n_iter=10,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'learning_rate': 0.1, 'iterations': 100, 'depth': 4}
Best CV accuracy: 0.5565
Test accuracy: 0.8784


In [27]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (CatBoost)
from catboost import CatBoostClassifier
clf_name = "CatBoost"
clf =   CatBoostClassifier(iterations=20, verbose=0)


# Store results
results7 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results7.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results7_df7 = pd.DataFrame(results7)

# Print
print(f"Results for {clf_name}:")
print(results7_df7)

# Save to CSV if needed
results7_df7.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for CatBoost:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   CatBoost  0.891245   0.924852  0.852468     0.930277  0.887186   
1     2   CatBoost  0.886731   0.923824  0.840652     0.931960  0.880278   
2     3   CatBoost  0.899726   0.927037  0.866612     0.932499  0.895807   
3     4   CatBoost  0.894528   0.917458  0.868263     0.921067  0.892183   
4     5   CatBoost  0.894665   0.928698  0.856522     0.933333  0.891151   
5     6   CatBoost  0.892613   0.921724  0.858547     0.926809  0.889015   
6     7   CatBoost  0.893830   0.935939  0.850922     0.938867  0.891408   
7     8   CatBoost  0.886168   0.920967  0.847030     0.926008  0.882453   
8     9   CatBoost  0.894514   0.924613  0.856946     0.931399  0.889494   
9    10   CatBoost  0.888220   0.917938  0.845938     0.928305  0.880468   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.890523  0.069723  0.891373  0.784966  0.782543           0.891

In [28]:
# Model Gradient Boosting

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier

model8 =   GradientBoostingClassifier(n_estimators=25)

param_dist8 = {
      "learning_rate": [0.1, 0.3,0.6],
      "n_estimators": [50, 100],

}

In [29]:

# Gradient Boosting

random_search = RandomizedSearchCV(model8, param_distributions=param_dist8, n_iter=10,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'n_estimators': 100, 'learning_rate': 0.1}
Best CV accuracy: 0.5797
Test accuracy: 0.8678


In [30]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Gradient Boosting)
from sklearn.ensemble import  GradientBoostingClassifier

clf_name = "Gradient Boosting"
clf =    GradientBoostingClassifier()


# Store results
results8 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results8.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results8_df8= pd.DataFrame(results8)

# Print
print(f"Results for {clf_name}:")
print(results8_df8)

# Save to CSV if needed
results8_df8.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)


Results for Gradient Boosting:
   Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Gradient Boosting  0.867715   0.907855  0.819471     0.916278   
1     2  Gradient Boosting  0.863885   0.898846  0.817178     0.909732   
2     3  Gradient Boosting  0.872777   0.902200  0.834708     0.910452   
3     4  Gradient Boosting  0.864159   0.891616  0.830702     0.897965   
4     5  Gradient Boosting  0.872914   0.910720  0.828804     0.917631   
5     6  Gradient Boosting  0.869083   0.898380  0.832878     0.905428   
6     7  Gradient Boosting  0.871391   0.913788  0.826877     0.918116   
7     8  Gradient Boosting  0.867697   0.900235  0.829672     0.906405   
8     9  Gradient Boosting  0.875359   0.899941  0.842033     0.908080   
9    10  Gradient Boosting  0.867834   0.901456  0.817824     0.915245   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.861402  0.866524  0.083722  0.867874  0.739044  0.735513   
1  0.856068  0.862214  0

In [31]:
# Model MLP

from sklearn.neural_network import MLPClassifier
model9 =   MLPClassifier(max_iter=300)
param_dist9 = {
      "hidden_layer_sizes": [(64,64), (128,64)]
}

In [32]:
# MLP

random_search = RandomizedSearchCV(model9, param_distributions=param_dist9, n_iter=10,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'hidden_layer_sizes': (128, 64)}
Best CV accuracy: 0.6877
Test accuracy: 0.8450


In [33]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (MLP)
from sklearn.neural_network import MLPClassifier
clf_name = "MLP"
clf =   MLPClassifier(max_iter=500)


# Store results
results9 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results9.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results9_df9= pd.DataFrame(results9)

# Print
print(f"Results for {clf_name}:")
print(results9_df9)

# Save to CSV if needed
results9_df9.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for MLP:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        MLP  0.809986   0.756301  0.916553     0.702718  0.828751   
1     2        MLP  0.835294   0.923292  0.727976     0.940634  0.814083   
2     3        MLP  0.829275   0.797311  0.880638     0.778443  0.836905   
3     4        MLP  0.835705   0.933707  0.724551     0.948020  0.815939   
4     5        MLP  0.850342   0.864842  0.832880     0.868044  0.848560   
5     6        MLP  0.848837   0.842303  0.859093     0.838542  0.850615   
6     7        MLP  0.738268   0.673427  0.949239     0.516826  0.787892   
7     8        MLP  0.841018   0.825975  0.867643     0.813915  0.846296   
8     9        MLP  0.857299   0.868918  0.838442     0.875813  0.853408   
9    10        MLP  0.841155   0.927552  0.730672     0.945896  0.817424   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.802545  0.297282  0.809635  0.634221  0.619703           0.809635  

In [34]:
# Model Bagging
from sklearn.ensemble import BaggingClassifier
model10 =   BaggingClassifier(n_estimators=10)
param_dist10 = {
     "n_estimators": [50, 100]
}


In [35]:


# BaggingClassifier

random_search = RandomizedSearchCV(model10, param_distributions=param_dist10, n_iter=10,
                                   cv=5, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best parameters: {'n_estimators': 50}
Best CV accuracy: 0.5334
Test accuracy: 0.9330


In [36]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (BaggingClassifier)
from sklearn.ensemble import BaggingClassifier
clf_name = "BaggingClassifier"
clf =   BaggingClassifier(n_estimators=10)


# Store results
results10 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results10.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results10_df10= pd.DataFrame(results10)

# Print
print(f"Results for {clf_name}:")
print(results10_df10)

# Save to CSV if needed
results10_df10.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for BaggingClassifier:
   Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  BaggingClassifier  0.929822   0.936600  0.922552     0.937140   
1     2  BaggingClassifier  0.922982   0.929736  0.913560     0.932231   
2     3  BaggingClassifier  0.937346   0.942126  0.931243     0.943386   
3     4  BaggingClassifier  0.930917   0.933753  0.928416     0.933443   
4     5  BaggingClassifier  0.932011   0.941715  0.922011     0.942149   
5     6  BaggingClassifier  0.934747   0.941503  0.927362     0.942160   
6     7  BaggingClassifier  0.932823   0.944992  0.922522     0.943634   
7     8  BaggingClassifier  0.928171   0.933388  0.923515     0.932910   
8     9  BaggingClassifier  0.934601   0.934236  0.933720     0.935466   
9    10  BaggingClassifier  0.931591   0.936839  0.921563     0.941098   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.929523  0.929818  0.062860  0.929846  0.859747  0.859650   
1  0.921577  0.922848  0

In [51]:
# Model Stacking
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
base_models = [
    ('lr', LogisticRegression(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42)),
]

#  Initialize StackingClassifier with the 'estimators' argument
model11 = StackingClassifier(
    estimators=base_models,
    cv=3,
    passthrough=True,
    n_jobs=-1,
    verbose=1,
)

param_dist11 = {
    "final_estimator": [
        LogisticRegression(random_state=42),
        RandomForestClassifier(random_state=42),
        DecisionTreeClassifier(random_state=42)
    ],
    "passthrough": [True, False]
    }




In [52]:

# StackingClassifier

random_search = RandomizedSearchCV(model11, param_distributions=param_dist11, n_iter=5,
                                   cv=3, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))



/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'passthrough': False, 'final_estimator': LogisticRegression(random_state=42)}
Best CV accuracy: 0.5919
Test accuracy: 0.3020


In [57]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (StackingClassifier)
from sklearn.ensemble import StackingClassifier
clf_name = "StackingClassifier"
clf =  best_model_LR


# Store results
results11 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results11.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results11_df11= pd.DataFrame(results11)

# Print
print(f"Results for {clf_name}:")
print(results11_df11)

# Save to CSV if needed
results11_df11.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for StackingClassifier:
   Fold          Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  StackingClassifier  0.295349   0.371181  0.583038     0.005764   
1     2  StackingClassifier  0.292613   0.365778  0.583264     0.007319   
2     3  StackingClassifier  0.196717   0.280094  0.391639     0.003811   
3     4  StackingClassifier  0.164432   0.246669  0.322537     0.004675   
4     5  StackingClassifier  0.163064   0.245936  0.320652     0.003306   
5     6  StackingClassifier  0.310397   0.382838  0.615238     0.004386   
6     7  StackingClassifier  0.302230   0.381733  0.585092     0.005328   
7     8  StackingClassifier  0.302641   0.378280  0.594250     0.005798   
8     9  StackingClassifier  0.316049   0.383929  0.629384     0.008406   
9    10  StackingClassifier  0.301956   0.369090  0.612314     0.007729   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.453591  0.057973  0.994236  0.294401 -0.502989 -0.411972   
1  0.449601 

In [58]:
# Combine all results into ONE Excel file (multiple sheets)

with pd.ExcelWriter("Purbasa_Bhowmick_SMOTE.xlsx") as writer:

    results_df.to_excel(writer, sheet_name="Logistic Regression", index=False)
    results1_df1.to_excel(writer, sheet_name="Decision Tree", index=False)
    results2_df2.to_excel(writer, sheet_name="Random Forest", index=False)
    results3_df3.to_excel(writer, sheet_name="KNN", index=False)
    results4_df4.to_excel(writer, sheet_name="SVM", index=False)
    results5_df5.to_excel(writer, sheet_name="XGBoost", index=False)
    results6_df6.to_excel(writer, sheet_name="LightGBM", index=False)
    results7_df7.to_excel(writer, sheet_name="CatBoost", index=False)
    results8_df8.to_excel(writer, sheet_name="Gradient Boosting", index=False)
    results9_df9.to_excel(writer, sheet_name="MLP", index=False)
    results10_df10.to_excel(writer, sheet_name="Bagging", index=False)
    results11_df11.to_excel(writer, sheet_name="Stacking", index=False)


In [59]:

from google.colab import files
files.download("Purbasa_Bhowmick_SMOTE.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>